# 条形图的几种形式

> 对应代码：`MathStatsCode/code_in_notes/Chap.7/charts_bar.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：演示 graph bar / graph hbar 的频数、均值条形图，以及均值+中位数+频数的组合图。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.7`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.7")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

这一个代码段演示了 `graph bar` 系列命令的四种常见形式，并用手工汇总数据画出组合图：

**①频数条形图**：`graph bar, over(a2012)` 默认对每个学历组画**均值**（不过没有指定被统计变量时，画的是各组观测数即频数）；`graph bar (count), over(a2012)` 用 `(count)` 统计量明确指定画频数。`over()` 为分组变量，`label(angle(15))` 让分类标签倾斜15度避免重叠。

**②均值条形图**：`graph hbar (mean) labor_inc, over(a2012)`，`hbar` 表示横向条形图，`(mean)` 指定统计量为组内均值。

**③组合条形图**：先 `collapse (count)/(mean)/(median), by(a2012)` 把数据聚合成“每个学历一组”的汇总数据（原数据被替换），再构造左右错开的横坐标（`edu_left=a2012-0.2`、`edu_right=a2012+0.2`），用三个 `bar` 图层在错开位置上分别画收入均值、收入中位数与人数；`yaxis(2)` 让人数使用**右侧第二纵轴**（金额与人数量纲不同），`barw(0.2)` 控制条宽，`legend` 用 `row(3)` 把图例排成三列放在图内。多重图层的错位/双轴技巧在画“汇总对比图”时非常常用。


In [ ]:
%%stata
use ../datasets/chfs_ind.dta, clear
label def education 1 "文盲"
label def education 2 "小学", add
label def education 3 "初中", add
label def education 4 "高中", add
label def education 5 "中专/职高", add
label def education 6 "大专/高职", add
label def education 7 "大学本科", add
label def education 8 "硕士研究生", add
label def education 9 "博士研究生", add
label values a2012 education
graph bar, over(a2012, label(angle(15)))
graph bar (count), over(a2012, label(angle(15)))
graph hbar (mean) labor_inc, over(a2012)
collapse (count) labor_inc (mean) mean_income = labor_inc (median) median_income = labor_inc, by(a2012)
gen edu_left=a2012-0.2
gen edu_right=a2012+0.2
twoway (bar mean_income edu_left, barw(0.2)) (bar median_income a2012, barw(0.2)) (bar labor_inc edu_right, yaxis(2) barw(0.2)), xlabel(1 "文盲" 2 "小学" 3 "初中" 4 "高中" 5 "中专/职高'" 6 "大专/高职" 7 "大学本科" 8 "硕士研究生" 9 "博士研究生", angle(20)) xtitle("文化程度") ytitle("金额", axis(1)) ytitle("人数", axis(2)) legend(pos(11) ring(0) label(1 "收入均值") label(2 "收入中位数") lab(3 "频数") row(3) symxsize(4))
